# 07 - Fuzzy AHP Priority Ranking Design

This notebook documents the Phase 10A Fuzzy AHP expert judgement framework for SentiRank. It does not calculate Fuzzy AHP weights, defuzzified scores, or final rankings in this phase.


## Phase 10B Backend Availability

Backend Fuzzy AHP calculation is available through `POST /ahp/fuzzy-calculate`. This notebook still does not run final expert judgement calculation; final Fuzzy AHP weights should be generated only after validated expert inputs are available.


## Fuzzy AHP Role In SentiRank

Fuzzy AHP will be compared with standard AHP because expert importance judgements can be uncertain. It uses linguistic scales and triangular fuzzy numbers to represent that uncertainty before later defuzzification and ranking.


In [ ]:
from pathlib import Path
import json

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "ml-service").exists()
            and (candidate / "docs").exists()
            and (candidate / "datasets").exists()
        ):
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = find_project_root()
TEMPLATE_DIR = PROJECT_ROOT / "docs" / "templates" / "ahp"
CRITERIA_JSON = TEMPLATE_DIR / "final_criteria_for_ahp.json"
FUZZY_TEMPLATE_CSV = TEMPLATE_DIR / "fuzzy_ahp_pairwise_template.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TEMPLATE_DIR:", TEMPLATE_DIR)


def load_json(path: Path):
    if not path.exists():
        print(f"Missing JSON: {path.relative_to(PROJECT_ROOT)}")
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing CSV: {path.relative_to(PROJECT_ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path)


## Final Criteria

Fuzzy AHP uses the same five candidate criteria as AHP. The criteria are derived from the selected SVM `merged_5class` taxonomy and still require expert judgement validation before final weighting.


In [ ]:
criteria_payload = load_json(CRITERIA_JSON)
if criteria_payload:
    criteria_df = pd.DataFrame(criteria_payload.get("criteria", []))
    display(criteria_df)


## Linguistic Scale And Triangular Fuzzy Numbers

The Phase 10A template uses these triangular fuzzy numbers: `equal = (1, 1, 1)`, `moderate = (2, 3, 4)`, `strong = (4, 5, 6)`, `very_strong = (6, 7, 8)`, and `extreme = (8, 9, 9)`. If `criterion_b` is preferred over `criterion_a`, the reciprocal TFN is `(1/u, 1/m, 1/l)`.


## Fuzzy Pairwise Template

The template pre-fills only `comparison_id`, `criterion_a`, and `criterion_b`; expert answer fields remain blank.


In [ ]:
fuzzy_template_df = load_csv(FUZZY_TEMPLATE_CSV)
if not fuzzy_template_df.empty:
    print("Pairwise comparisons:", len(fuzzy_template_df))
    display(fuzzy_template_df)


## Expected Later Output

Later phases may create Fuzzy AHP weights, defuzzified priority scores, and rankings. Phase 10A creates only criteria definitions and expert judgement templates.
